In [ ]:
import pygame
pygame.init()

screen = pygame.display.set_mode((800, 600))
pygame.display.set_caption("sprites")

player_img = pygame.image.load("zelda.png")
player_img = pygame.transform.scale(player_img, (50, 50))
player_pos = [100, 100]

running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    

    screen.fill((255, 255, 255))
    screen.blit(player_img, player_pos)

    player_rect = pygame.Rect(*player_pos, 50, 50)
    pygame.draw.rect(screen, (0,255,0), player_rect, width=3)

    collision = player_rect.colliderect(enemy_rect)
    if collision:
        pygame.draw.rect(screen, (255,0,0), player_rect, width=3)

    pygame.display.flip()

pygame.quit()

In [ ]:
import pygame
import random
import sys
pygame.init()
 
screen = pygame.display.set_mode((800,600)) #width, height
pygame.display.set_caption("Advanced Collision Detection")
 
speed = 5
enemy_speed = 3
lives = 3
font = pygame.font.SysFont("Arial", 30)
game_duration = 30 * 1000 #30s in miliseconds
start_time = pygame.time.get_ticks()
 
#tickrate
clock = pygame.time.Clock()
 
PLAYER_WIDTH = 50
PLAYER_HEIGHT = 50
 
player_pos = [100,100]
enemy_pos = [500,300]
obstacle_pos = [150, 150]
obstacle_height = 100
obstacle_width = 100
 
#load images
player_img = pygame.image.load("zelda.png").convert_alpha()
enemy_img = pygame.image.load("ganon.png").convert_alpha()
#scale image
player_img = pygame.transform.scale(player_img, (PLAYER_WIDTH,PLAYER_HEIGHT))
enemy_img = pygame.transform.scale(enemy_img, (50,50))
 
#create masks - to keep visible pixels
player_mask = pygame.mask.from_surface(player_img)
enemy_mask = pygame.mask.from_surface(enemy_img)

#enemy movement direction
enemy_direction = [random.choice([-1,1]), random.choice([-1,1])]
 
running = True
game_over = False
 
while running:
    while game_over:
        #game over or win screen
        screen.fill((255,255,255))
        if lives > 0:
            result_text = font.render("YOU WIN! (somehow it was suppose to be impossible >:( ). Press 'C' to play again or 'Q' to quit.", True, (0,255,0))
        else:
            result_text = font.render("Don't give up! 99.9999999% of losers quit before they finally win! Press 'C' to play again or 'Q' to quit.", True, (255,0,0))
        final_score_text = font.render(f"Lives remaining: {lives}", True, (0,0,0))
        screen.blit(result_text, (screen.get_width()//2 - result_text.get_width()//2, 200))
        screen.blit(final_score_text, (screen.get_width()//2 -  final_score_text.get_width()//2, 250))
        pygame.display.flip()

    
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
                game_over = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q:
                    running = False
                    game_over = False
                if event.key == pygame.K_c:
                    game_over = False
                    player_pos = [100, 100]
                    enemy_pos = [300, 300]
                    lives = 3
                    start_time = pygame.time.get_ticks()
    #detect closing the window
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
 
    keys = pygame.key.get_pressed()
    if keys[pygame.K_LEFT]:
        player_pos[0] -= speed
    if keys[pygame.K_RIGHT]:
        player_pos[0] += speed
    if keys[pygame.K_UP]:
        player_pos[1] -= speed
    if keys[pygame.K_DOWN]:
        player_pos[1] += speed
 
    #time duration
    elapsed_time = pygame.time.get_ticks() - start_time
    remaining_time = max(0,(game_duration - elapsed_time) // 1000)
 
    #Win/Lose Condition
    if remaining_time == 0 or lives == 0:
        game_over = True
 
    #move the enemy
    enemy_pos[0] += enemy_direction[0] * enemy_speed
    enemy_pos[1] += enemy_direction[1] * enemy_speed
 
    #bounce the enemy off the screen edges
    if enemy_pos[0] < 0 or enemy_pos[0] + 100 >= 800:
        enemy_direction[0] *= -1
    if enemy_pos[1] < 0 or enemy_pos[1] + 100 >= 600:
        enemy_direction[1] *= -1
 
    #limit the player to the window
    if player_pos[0] < 0:
        player_pos[0] = 0
    elif player_pos[0] > 800 - PLAYER_WIDTH:
        player_pos[0] = 800 - PLAYER_WIDTH
    elif player_pos[1] < 0:
        player_pos[1] = 0
    elif player_pos[1] > 600 - PLAYER_HEIGHT:
        player_pos[1] = 600 - PLAYER_HEIGHT
 
    #Collision detection using masks
    player_rect = pygame.Rect(*player_pos,50,50)
    enemy_rect = pygame.Rect(*enemy_pos,100,100)
    #offset - where's enemy relative to player -horizontally & vertically
    offset = (enemy_rect.left - player_rect.left, enemy_rect.top - player_rect.top)
    #check 2 shapes touch pixel by pixel
    collision = player_mask.overlap(enemy_mask, offset) 
 
    #fill, blit, flip (order matters!)
    screen.fill((255,255,255))
    #blit- copy the pixels of one surface to another
    screen.blit(player_img, player_pos)
    screen.blit(enemy_img, enemy_pos)
 
    #draw outline of player image
    player_outline = player_mask.outline()
    player_outline = [(x + player_pos[0], y + player_pos[1]) for x,y in player_outline]
    pygame.draw.lines(screen,(0,255,0), True, player_outline, 3)
 
    #draw outline of enemy image
    enemy_outline = enemy_mask.outline()
    enemy_outline = [(x + enemy_pos[0], y + enemy_pos[1]) for x,y in enemy_outline]
    pygame.draw.lines(screen,(0,255,0), True, enemy_outline, 3)
 
    #handle collision    
    if collision:
        lives -= 1
        pygame.draw.lines(screen, (255,0,0), True, player_outline, width = 3)
        screen.blit(font.render("You got hit!", True, (255,0,0)), player_pos)
        pygame.display.flip()
        pygame.time.wait(1000)
        start_time += 1000
        player_pos = [random.randint(0,800), random.randint(0,600)]

    # safe zone
    obstacle_rect = pygame.Rect(*obstacle_pos, obstacle_width, obstacle_height)
    pygame.draw.rect(screen, (0, 255, 0), obstacle_rect, width = 3)
    obs_collision = obstacle_rect.colliderect(enemy_rect)
    if obs_collision:
        delta_x = obstacle_rect.centerx - enemy_rect.centerx
        delta_y = obstacle_rect.centery - enemy_rect.centery
        if abs(delta_x) > abs(delta_y):
            enemy_direction[0] *= -1
            enemy_speed *= 1.01
            random.random()
        else:
            enemy_direction[1] *= -1
        
        obstacle_width = random.randint(50, 150)
        obstacle_height = random.randint(50, 150)
        obstacle_pos = [random.randint(0, 800-obstacle_width), random.randint(0, 600-obstacle_height)]

    #Add display
    lives_text = font.render(f"Lives: {lives}", True, (0,0,0))
    time_text = font.render(f"Time: {remaining_time}s", True, (0,0,0))
    screen.blit(lives_text, (10,10))
    screen.blit(time_text, (10,50))
 
    pygame.display.flip()
    clock.tick(60)
 
pygame.quit()
sys.exit()

SystemExit: 